Geometria Riemanniana, graficos

In [6]:
import os
import glob
import numpy as np
import pandas as pd
import plotly.express as px
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

In [10]:
# 1. Definir ruta y buscar todos los archivos de sujetos (.npz)
data_dir = r"..\data\preprocessed"
file_paths = sorted(glob.glob(os.path.join(data_dir, "S*_preprocessed.npz")))

all_records = []
sqrt2 = np.sqrt(2.0)

# 2. Iterar sobre los 15 archivos de sujetos
for path in file_paths[:15]:
    subject_id = os.path.basename(path).split("_")[0]  # Extrae ej: "S01"
    data = np.load(path)
    
    # Extraer la variable 'x'
    X = data['x'] if 'x' in data else data[data.files[0]]  # Shape: (trials, 6, 512)
    
    # Seleccionar primer (0) y último (-1) canal -> (trials, 2, 512)
    X_sub = X[:, [0, -1], :]
    
    # Centrar en el tiempo por canal y trial
    X_centered = X_sub - np.mean(X_sub, axis=-1, keepdims=True)
    T = X_centered.shape[-1]
    
    # Calcular matriz de covarianza 2x2 por trial
    C = np.matmul(X_centered, np.swapaxes(X_centered, -1, -2)) / (T - 1)
    
    # Extraer triángulo superior escalado por sqrt(2) para términos no diagonales
    var_c1 = C[:, 0, 0]
    cov_12 = C[:, 0, 1] * sqrt2
    var_c2 = C[:, 1, 1]
    
    # Guardar registros ordenados con metadata
    for i in range(len(C)):
        all_records.append({
            'Sujeto': subject_id,
            'Trial': i,
            'Sigma_11': var_c1[i],
            'Sqrt2_Sigma_12': cov_12[i],
            'Sigma_22': var_c2[i]
        })

# Convertir a DataFrame
df = pd.DataFrame(all_records)

# 3. Graficar en 3D interactivo con Plotly
fig = px.scatter_3d(
    df,
    x='Sigma_11',
    y='Sqrt2_Sigma_12',
    z='Sigma_22',
    color='Sujeto',
    hover_data=['Sujeto', 'Trial'],
    title="Espacio 3D de Covarianzas (Primer y Último Canal) - Todos los Sujetos",
    labels={
        'Sigma_11': 'Var Canal 1 (Σ11)',
        'Sqrt2_Sigma_12': '√2 Cov Cruzada (√2 Σ12)',
        'Sigma_22': 'Var Canal 2 (Σ22)'
    }
)

# Ajustes de estilo para mejorar la rotación y visibilidad
fig.update_traces(marker=dict(size=2, opacity=0.7))
# Definí los límites [mínimo, máximo] que desees para cada variable
fig.update_layout(
    scene=dict(
        aspectmode='cube',
        xaxis=dict(
            title='Var C1 (Σ11)',
            range=[0, 1000]  # Límites min y max para el eje X
        ),
        yaxis=dict(
            title='√2 Cov (√2 Σ12)',
            range=[-100, 400]  # Límites min y max para el eje Y
        ),
        zaxis=dict(
            title='Var C2 (Σ22)',
            range=[0, 1000]  # Límites min y max para el eje Z
        )
    )
)

fig.show()

In [9]:
# 1. Buscar todos los archivos de los sujetos (.npz)
data_dir = r"..\data\preprocessed"
file_paths = sorted(glob.glob(os.path.join(data_dir, "S*_preprocessed.npz")))

cov_list = []
subject_labels = []
trial_labels = []

# 2. Cargar y calcular matrices de covarianza (2x2) con pyriemann / numpy
for path in file_paths[:15]:
    subject_id = os.path.basename(path).split("_")[0]
    data = np.load(path)
    X = data['x'] if 'x' in data else data[data.files[0]]  # (trials, 6, 512)
    
    # Quedarse con el primer (0) y último (-1) canal
    X_sub = X[:, [0, -1], :]
    
    # Estimar covarianzas (por defecto aplica centrado temporal y regularización SPD)
    # Forma resultante: (trials, 2, 2)
    covs = Covariances(estimator='lwf').transform(X_sub)
    
    for i in range(len(covs)):
        cov_list.append(covs[i])
        subject_labels.append(subject_id)
        trial_labels.append(i)

# Matriz de covarianzas de todos los trials: (N_total, 2, 2)
C_all = np.array(cov_list)

# 3. Proyección al Espacio Tangente (Tangent Space)
# Se calcula la media de Fréchet sobre la variedad y se mapea cada matriz a R^3
ts = TangentSpace(metric='riemann')
X_ts = ts.fit_transform(C_all)  # Genera directamente los 3 componentes escalados con sqrt(2)

# X_ts tiene forma (N_total, 3): [S11, sqrt(2)*S12, S22]

# 4. Crear DataFrame con las características en el espacio tangente
df_ts = pd.DataFrame({
    'Sujeto': subject_labels,
    'Trial': trial_labels,
    'S_11': X_ts[:, 0],
    'Sqrt2_S_12': X_ts[:, 1],
    'S_22': X_ts[:, 2]
})

# 5. Visualizar las covarianzas proyectadas en el Espacio Tangente 3D
fig = px.scatter_3d(
    df_ts,
    x='S_11',
    y='Sqrt2_S_12',
    z='S_22',
    color='Sujeto',
    hover_data=['Sujeto', 'Trial'],
    title="Covarianzas Proyectadas en el Espacio Tangente Riemanniano (T_Cref)",
    labels={
        'S_11': 'Tangente S11',
        'Sqrt2_S_12': '√2 Tangente S12',
        'S_22': 'Tangente S22'
    }
)

# Ajustes gráficos de la caja y perspectiva
fig.update_traces(marker=dict(size=2, opacity=0.7))
fig.update_layout(
    scene=dict(
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=1),  # Caja cúbica
        xaxis_title='S11 (Eje Tangente 1)',
        yaxis_title='√2 S12 (Eje Tangente 2)',
        zaxis_title='S22 (Eje Tangente 3)'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()